In [ ]:
import cv2
import mediapipe as mp
import numpy as np

# Mediapipe
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Cam
cap = cv2.VideoCapture(0)
canvas = None

prev_x, prev_y = 0, 0  # Previous Finger Tip X,Y

# Color
color = (0, 0, 255)  # Red

with mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7) as hands:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape

        if canvas is None:
            canvas = np.zeros_like(frame)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                # Definition of finger 
                thumb_tip = hand_landmarks.landmark[8]
                x, y = int(thumb_tip.x * w), int(thumb_tip.y * h)

                # Drawing a circle
                cv2.circle(frame, (x, y), 5, color, -1)

                # if there is previous finger position draw
                if prev_x != 0 and prev_y != 0:
                    cv2.line(canvas, (prev_x, prev_y), (x, y), color, 5)

                # Save current finger position
                prev_x, prev_y = x, y

        # Overlaying the canvas
        frame = cv2.addWeighted(frame, 1, canvas, 0.5, 0)

        cv2.imshow("Air Canvas", frame)

        key = cv2.waitKey(1)
        if key & 0xFF == ord('q'):
            break
        elif key & 0xFF == ord('c'):
            canvas = None  # Clear the canvas

cap.release()
cv2.destroyAllWindows()
